[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabin2004/Machine-Learning-Bootcamp/blob/main/Module_05_Classification/02_knn.ipynb)

# Episode 12 – K-Nearest Neighbors (KNN)

**Machine Learning Bootcamp** | Module 05

---

## 🎯 Learning Objectives
- Understand the KNN algorithm and how distance is used
- Train a KNN classifier with scikit-learn
- Visualise decision boundaries and choose the optimal K

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

sns.set_theme(style='whitegrid')

## 1. How KNN Works

1. Store all training points.
2. For a new point, find the **K nearest** training points (by Euclidean distance).
3. Take the **majority vote** of their labels (classification) or average (regression).

No training step — KNN is a **lazy learner**.

In [ ]:
iris = load_iris()
X, y = iris.data[:, :2], iris.target  # Use first 2 features for visualization

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Decision boundaries for K = 1, 5, 15
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

xx, yy = np.meshgrid(
    np.linspace(X_train_s[:, 0].min() - 0.5, X_train_s[:, 0].max() + 0.5, 200),
    np.linspace(X_train_s[:, 1].min() - 0.5, X_train_s[:, 1].max() + 0.5, 200)
)

for ax, k in zip(axes, [1, 5, 15]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='tab10')
    ax.scatter(X_train_s[:, 0], X_train_s[:, 1], c=y_train, cmap='tab10', edgecolors='k', s=25)
    acc = accuracy_score(y_test, knn.predict(X_test_s))
    ax.set_title(f'K={k} | Test Acc={acc:.2%}')

plt.tight_layout(); plt.show()

## 2. Choosing the Optimal K with Cross-Validation

In [ ]:
# Use all 4 features this time
X_full, y_full = iris.data, iris.target
X_full_s = StandardScaler().fit_transform(X_full)

k_values = range(1, 31)
cv_scores = [
    cross_val_score(KNeighborsClassifier(n_neighbors=k), X_full_s, y_full, cv=5).mean()
    for k in k_values
]

best_k = k_values[np.argmax(cv_scores)]
plt.figure(figsize=(9, 5))
plt.plot(k_values, cv_scores, marker='o', color='steelblue')
plt.axvline(best_k, color='tomato', ls='--', label=f'Best K={best_k}')
plt.xlabel('K'); plt.ylabel('5-Fold CV Accuracy')
plt.title('Choosing K via Cross-Validation')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Best K: {best_k}  |  CV Accuracy: {max(cv_scores):.4f}')

## 🏋️ Exercises

1. Try `metric='manhattan'` instead of `'euclidean'`. Does the best K change?
2. What happens to accuracy if you do NOT scale the features? Explain why.
3. Apply KNN to the Breast Cancer dataset and compare accuracy with Logistic Regression.

---
**Next ▶ [Episode 13 – Decision Trees](03_decision_trees.ipynb)**